<a href="https://colab.research.google.com/github/piaseckazaneta/Python/blob/main/Witamy_w%C2%A0Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install geopy pyproj openpyxl

In [4]:
from google.colab import files
uploaded = files.upload()

Saving schroniska_polska.xlsx to schroniska_polska (1).xlsx


In [5]:
"""
Skrypt do geokodowania schronisk i dodania współrzędnych EPSG:2180 do pliku Excel.

WYMAGANIA:
    pip install geopy pyproj openpyxl

UŻYCIE:
    python dodaj_wspolrzedne.py

Skrypt automatycznie wczyta plik 'schroniska_polska.xlsx' z tego samego folderu
i zapisze wynik jako 'schroniska_polska_geo.xlsx'.

Czas działania: ~4-5 minut (Nominatim wymaga 1s przerwy między zapytaniami).
"""

import time
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
from pyproj import Transformer

# ── konfiguracja ──────────────────────────────────────────────────────────────

INPUT_FILE  = "schroniska_polska.xlsx"
OUTPUT_FILE = "schroniska_polska_geo.xlsx"
SHEET_NAME  = "Schroniska w Polsce"

# transformacja WGS84 → EPSG:2180 (układ 1992, Polska)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:2180", always_xy=True)
geolocator  = Nominatim(user_agent="schroniska_wdf_geocoder_v1", timeout=10)

# ── style ─────────────────────────────────────────────────────────────────────

HEADER_FILL  = PatternFill("solid", start_color="2C5F2E")
HEADER_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=11)
HEADER_ALIGN = Alignment(horizontal="center", vertical="center", wrap_text=True)
CELL_FONT    = Font(name="Arial", size=10)
CELL_ALIGN   = Alignment(vertical="center")
ALT_FILL     = PatternFill("solid", start_color="F0F7F0")
thin         = Side(style="thin", color="CCCCCC")
BORDER       = Border(left=thin, right=thin, top=thin, bottom=thin)

# ── geokodowanie ──────────────────────────────────────────────────────────────

def geocode(street: str, postcode: str, city: str, voivodeship: str, city_cache: dict):
    """Zwraca (lat, lon) lub (None, None). Używa cache dla miast."""

    queries = []
    if street and postcode:
        queries.append(f"{street}, {postcode} {city}, Poland")
    if street and city:
        queries.append(f"{street}, {city}, Poland")

    # zapytanie po mieście – sprawdź cache
    city_key = f"{city}_{voivodeship}"
    if city_key not in city_cache:
        try:
            loc = geolocator.geocode(f"{city}, woj. {voivodeship}, Poland",
                                     country_codes="pl")
            time.sleep(1.1)
            city_cache[city_key] = (loc.latitude, loc.longitude) if loc else (None, None)
        except (GeocoderTimedOut, GeocoderServiceError):
            time.sleep(2.0)
            city_cache[city_key] = (None, None)

    queries.append(city_key)   # sentinel – obsłużone niżej

    for q in queries:
        if q == city_key:
            lat, lon = city_cache[city_key]
            if lat:
                return lat, lon
            continue
        try:
            loc = geolocator.geocode(q, country_codes="pl")
            time.sleep(1.1)
            if loc:
                return loc.latitude, loc.longitude
        except (GeocoderTimedOut, GeocoderServiceError):
            time.sleep(2.0)

    return None, None


# ── główna logika ─────────────────────────────────────────────────────────────

def main():
    print(f"Wczytuję: {INPUT_FILE}")
    wb = openpyxl.load_workbook(INPUT_FILE)
    ws = wb[SHEET_NAME]

    # nowe nagłówki kolumn 11–14
    new_headers = {
        11: "X (EPSG:2180)",
        12: "Y (EPSG:2180)",
        13: "Lon (WGS84)",
        14: "Lat (WGS84)",
    }
    for col, label in new_headers.items():
        cell = ws.cell(row=1, column=col, value=label)
        cell.fill  = HEADER_FILL
        cell.font  = HEADER_FONT
        cell.alignment = HEADER_ALIGN
        cell.border = BORDER
        ws.column_dimensions[get_column_letter(col)].width = 16

    # zaktualizuj zakres auto_filtra
    last_col = get_column_letter(14)
    ws.auto_filter.ref = f"A1:{last_col}{ws.max_row}"

    city_cache: dict = {}
    total    = ws.max_row - 1
    success  = 0
    failed   = []

    print(f"Geokuję {total} schronisk (ok. {total * 1.1 / 60:.1f} min)…\n")

    for row in range(2, ws.max_row + 1):
        name       = ws.cell(row=row, column=2).value  or ""
        city       = ws.cell(row=row, column=3).value  or ""
        street     = ws.cell(row=row, column=4).value  or ""
        postcode   = ws.cell(row=row, column=5).value  or ""
        voivodeship= ws.cell(row=row, column=7).value  or ""
        is_even    = (row % 2 == 0)

        print(f"  [{row-1:>3}/{total}] {city:<25} {street[:30]}", end=" … ", flush=True)

        lat, lon = geocode(street, postcode, city, voivodeship, city_cache)

        if lat and lon:
            x, y = transformer.transform(lon, lat)
            ws.cell(row=row, column=11, value=round(x, 2))
            ws.cell(row=row, column=12, value=round(y, 2))
            ws.cell(row=row, column=13, value=round(lon, 6))
            ws.cell(row=row, column=14, value=round(lat, 6))
            success += 1
            print(f"OK  ({round(x):,} / {round(y):,})")
        else:
            ws.cell(row=row, column=11, value="BRAK")
            ws.cell(row=row, column=12, value="BRAK")
            ws.cell(row=row, column=13, value="BRAK")
            ws.cell(row=row, column=14, value="BRAK")
            failed.append(f"  wiersz {row}: {city} – {name}")
            print("BRAK wyniku")

        # styl komórek
        for col in range(11, 15):
            cell = ws.cell(row=row, column=col)
            cell.font   = CELL_FONT
            cell.alignment = CELL_ALIGN
            cell.border = BORDER
            if is_even:
                cell.fill = ALT_FILL

    # podsumowanie
    print(f"\n{'='*55}")
    print(f"Geokodowane:  {success}/{total}")
    print(f"Brak wyniku:  {len(failed)}")
    if failed:
        print("Rekordy bez współrzędnych:")
        for f in failed:
            print(f)

    wb.save(OUTPUT_FILE)
    print(f"\nPlik zapisany: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Wczytuję: schroniska_polska.xlsx
Geokuję 149 schronisk (ok. 2.7 min)…

  [  1/149] Gdynia                    ul. Małokacka 3A … OK  (474,379 / 740,009)
  [  2/149] Tczew                     ul. Malinowska … OK  (485,480 / 691,857)
  [  3/149] Luzino                    Dąbrówka Młyn 30 … OK  (446,532 / 745,072)
  [  4/149] Starogard Gdański         ul. Hermanowska 24 … OK  (468,907 / 679,288)
  [  5/149] Elbląg                    ul. Królewiecka 233 … OK  (528,766 / 703,939)
  [  6/149] Sompolno                  ul. Leśna 53 … OK  (466,188 / 502,617)
  [  7/149] Dzierżoniów               ul. Brzegowa 151 … OK  (334,679 / 320,640)
  [  8/149] Jelenia Góra              ul. Spółdzielcza 33a … OK  (264,537 / 336,801)
  [  9/149] Legnica                   ul. Ceglana 1 … OK  (302,698 / 374,687)
  [ 10/149] Oleśnica                  Ligota Nowa 19 … OK  (388,697 / 363,735)
  [ 11/149] Pieńsk                    Dłużyna Górna 1F … OK  (234,004 / 381,718)
  [ 12/149] Wałbrzych                 ul

In [6]:
files.download("schroniska_polska_geo.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>